# Exercise 4.3 — The decision threshold (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day4/day4-03-thresholds.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

In Exercise 4.2 every threshold gave a different line, and 0.5, where both
classes are equally likely, was only the default. Why would anyone move it?
This exercise answers that question, back on one measurement, age.

A logistic regression outputs a probability,

$$\hat{p}(x) = P(Y = 1 \mid x),$$

but a clinic needs a decision: offer this person treatment to prevent a heart
attack or stroke, or not. To turn
the probability into a predicted class we choose a **decision threshold**.
Usually:

- if $\hat{p}(x) \geq 0.5$, predict 1 (an event);
- if $\hat{p}(x) < 0.5$, predict 0 (no event).

But 0.5 is arbitrary. Changing the threshold changes how many true positives,
false positives, true negatives and false negatives the model makes, and so
which mistakes it makes.

**Theory:** decision threshold; confusion table; true and false positives and
negatives; sensitivity and specificity; why you would move the threshold.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

The same 500 people and the same split as Exercise 4.1. We add a
column `cvd_dummy`, the outcome coded 1 for `yes` and 0 for `no`, which
makes the counting below easier to read.

In [ ]:
cvd = pd.read_csv("data/cvd_risk.csv")
cvd["cvd_dummy"] = (cvd["cvd"] == "yes").astype(int) # 1 = yes, 0 = no

train, test = train_test_split(cvd, test_size=0.2, random_state=42)

------------------------------------------------------------------------

### Part 1 — From a probability to a predicted class

Fit the age-only logistic regression from Exercise 4.1 on the training rows:

In [ ]:
log_fit = LogisticRegression(C=np.inf)
log_fit = log_fit.fit(train[["age"]], train["cvd_dummy"])

print("intercept:", round(log_fit.intercept_[0], 4))
print("slope:    ", round(log_fit.coef_[0, 0], 4))

To make it easy to follow, predict the probability of an event for 15 people
from the test set only: the patients in rows 22 to 36. Python counts from 0, so
those are positions 21 to 35.

In [ ]:
patients15 = test.iloc[21:36].copy()

prediction_15 = patients15.copy()
prediction_15["prob"] = log_fit.predict_proba(patients15[["age"]])[:, 1]

prediction_15[["age", "cvd_dummy", "prob"]].round(3)

Now turn each probability into a predicted class at three thresholds: 0.3, 0.5
and 0.7. A person is predicted to have an event (1) when their probability is
at or above the threshold:

In [ ]:
prediction_15["class_0.3"] = (prediction_15["prob"] >= 0.3).astype(int)
prediction_15["class_0.5"] = (prediction_15["prob"] >= 0.5).astype(int)
prediction_15["class_0.7"] = (prediction_15["prob"] >= 0.7).astype(int)

prediction_15[["age", "cvd_dummy", "prob",
               "class_0.3", "class_0.5", "class_0.7"]].round(3)

Here are the 15 predicted probabilities, coloured by what is actually true, with
the three thresholds as dashed lines. Everyone above a line is predicted to have
an event at that threshold. The points follow the S-shaped curve from
Exercise 4.1.

In [ ]:
has_event = prediction_15["cvd_dummy"] == 1

fig, ax = plt.subplots()
ax.scatter(prediction_15.loc[has_event, "age"], prediction_15.loc[has_event, "prob"],
           color="darkorange", s=60, label="event (1)")
ax.scatter(prediction_15.loc[~has_event, "age"], prediction_15.loc[~has_event, "prob"],
           color="grey", s=60, label="no event (0)")
ax.axhline(0.3, color="blue",   linestyle="--")
ax.axhline(0.5, color="red",    linestyle="--")
ax.axhline(0.7, color="purple", linestyle="--")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Predicted probability of an event")
ax.set_title("Predicted probabilities with the thresholds 0.3, 0.5 and 0.7")
ax.legend(title="What is true")
plt.show()

> **Learning points**
>
> - A classifier gives a probability; a threshold turns it into a predicted class.
> - The threshold is a choice, not part of the model. 0.5 is only a convention.
> - Lowering the threshold predicts an event for more people; raising it, for fewer.

------------------------------------------------------------------------

### Part 2 — Confusion tables, sensitivity and specificity

Each person ends up in one of four boxes, depending on what is true and what the
model predicted:

| | Predicted 1 (event) | Predicted 0 (no event) |
|---|:-:|:-:|
| **Truly 1** | True positive (**TP**) | False negative (**FN**): a case missed |
| **Truly 0** | False positive (**FP**): a false alarm | True negative (**TN**) |

This is the **confusion table**. **On paper first:** use the table of 15 people
above to count TP, FP, FN and TN at the threshold 0.5. Then check with code.

In [ ]:
pd.crosstab(prediction_15["cvd_dummy"], prediction_15["class_0.5"],
            rownames=["True"], colnames=["Predicted"])

At 0.5 there are 3 true positives, 3 false positives, 4 false negatives and 5
true negatives. Now count all four boxes for the three thresholds side by side.
Two summaries of the table are used everywhere in medicine:

$$\text{Sensitivity} = \frac{TP}{TP + FN} \qquad\qquad
\text{Specificity} = \frac{TN}{TN + FP}$$

- **Sensitivity**: of the people who really had an event, the proportion the
  model catches.
- **Specificity**: of the people who really do not, the proportion the model
  correctly leaves alone.

Complete the code so that it counts the false negatives and computes the
specificity:

> **Stuck?**
>
> A false negative is a person who truly had an event (`truth == 1`) but was
> predicted not to (`predicted == 0`). Specificity uses the TN and FP columns, in
> the same way as sensitivity uses TP and FN.

> **Worked answer**

In [ ]:
rows = []
for threshold in ["0.3", "0.5", "0.7"]:
    truth     = prediction_15["cvd_dummy"]
    predicted = prediction_15["class_" + threshold]
    rows.append({"threshold": threshold,
                 "TP": ((truth == 1) & (predicted == 1)).sum(),
                 "FP": ((truth == 0) & (predicted == 1)).sum(),
                 "FN": ((truth == 1) & (predicted == 0)).sum(),
                 "TN": ((truth == 0) & (predicted == 0)).sum()})

threshold_compare = pd.DataFrame(rows)
threshold_compare["sensitivity"] = threshold_compare["TP"] / (threshold_compare["TP"] + threshold_compare["FN"])
threshold_compare["specificity"] = threshold_compare["TN"] / (threshold_compare["TN"] + threshold_compare["FP"])

threshold_compare.round(3)

> | Threshold | TP | FP | FN | TN | Sensitivity | Specificity |
> |:-:|:-:|:-:|:-:|:-:|:-:|:-:|
> | 0.3 | 6 | 6 | 1 | 2 | 0.857 | 0.250 |
> | 0.5 | 3 | 3 | 4 | 5 | 0.429 | 0.625 |
> | 0.7 | 1 | 0 | 6 | 8 | 0.143 | 1.000 |

Changing the threshold changes the kind of mistakes the model makes. A **low**
threshold catches more of the people who had an event (higher sensitivity) at the
price of more false alarms. A **high** threshold raises fewer false alarms
(higher specificity) but misses more cases. At 0.7 this model misses six of the
seven people who had an event.

> **Learning points**
>
> - The confusion table counts true positives, false positives, false negatives and true negatives.
> - Sensitivity = TP / (TP + FN); specificity = TN / (TN + FP).
> - Lowering the threshold raises sensitivity and lowers specificity, and the other way round.

------------------------------------------------------------------------

### Part 3 — The packaged version, on all 100 test patients

Fifteen people are enough to count by hand, not to judge a model. Whenever you
report a classification model you will build a confusion table, and packages
already do it for you. In scikit-learn it is `confusion_matrix()`:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

test_preds = test.copy()
test_preds["prob"] = log_fit.predict_proba(test[["age"]])[:, 1]
test_preds["predicted_0.5"] = (test_preds["prob"] >= 0.5).astype(int)

cm = confusion_matrix(test_preds["cvd_dummy"], test_preds["predicted_0.5"])
cm

In [ ]:
display = ConfusionMatrixDisplay(cm, display_labels=["no (0)", "yes (1)"])
display = display.plot(cmap="Blues")
plt.show()

> **Rows and columns are the other way round in R**
>
> `confusion_matrix()` puts the **truth** in the rows and the **prediction** in
> the columns. tidymodels' `conf_mat()` does the opposite. The numbers are the
> same; read the labels before you read the table.

At 0.5 the model finds 13 of the 31 people who had an event (sensitivity
0.419) and correctly leaves alone 54 of the 69 without (specificity 0.783). For
prevention, missing more than half of the people who go on to have a heart attack or
stroke is a poor result.

**Why move the threshold?** The two mistakes are not equally bad. A false
negative is a person who was not offered treatment and then had a heart attack
or stroke. A false positive is a person offered a tablet and a review
appointment they did not need. A clinic that wants to miss fewer cases lowers
the threshold. Compare three thresholds on all 100 test patients:

In [ ]:
had_event = test_preds["cvd_dummy"] == 1
for threshold in [0.3, 0.5, 0.7]:
    called_yes = test_preds["prob"] >= threshold
    sensitivity = (called_yes & had_event).sum() / had_event.sum()
    specificity = (~called_yes & ~had_event).sum() / (~had_event).sum()
    print(threshold, "sensitivity:", round(sensitivity, 3), "specificity:", round(specificity, 3))

At 0.3 the model finds 20 of the 31 people who had an event (sensitivity
0.645) but leaves alone only 35 of the 69 without (specificity 0.507). At 0.7
it leaves alone 68 of the 69 (specificity 0.986) but finds only 4 of the 31
(sensitivity 0.129). Lowering the threshold buys sensitivity with specificity;
raising it does the opposite. No threshold is best for every purpose: which
mistake matters more depends on the decision, and that is a clinical choice,
not a statistical one.

**Question!** Would you move the threshold up or down for a cheap, safe blood
test that only decides who gets a second, more accurate test? And for starting
a treatment with serious side effects?

> **Learning points**
>
> - Packages build the confusion table for you: `conf_mat()` in R, `confusion_matrix()` in Python. Check which way round the rows and columns are.
> - Lowering the threshold raises sensitivity and lowers specificity; raising it does the opposite. Which mistake matters more is a clinical decision, not a statistical one.
> - Report the threshold together with sensitivity and specificity: a threshold on its own says nothing.

## Stretch

> **Stretch — choose the threshold honestly**
>
> A clinic wants to find at least 90% of the people who will have an event. Find
> the highest threshold, in steps of 0.05, that reaches a sensitivity of 0.9 on
> the **training** rows. Then apply it once to the test patients: what
> sensitivity and specificity do you get there? Why is this the honest way round?